# 03 - Inference

Three equivalent ways to use the trained model:
1. the persisted pipeline `models/model.joblib` (what the Docker image serves),
2. the MLflow model registry (`models:/heart-disease-classifier@champion`),
3. the REST API (`POST /predict`) - local uvicorn, Docker or Kubernetes.

In [1]:
import json

import mlflow
import numpy as np
import pandas as pd
import requests

from heart_disease import config
from heart_disease.predict import load_metadata, load_model, predict_records

patients = [
    json.loads((config.PROJECT_ROOT / "sample_request.json").read_text()),
    json.loads((config.PROJECT_ROOT / "sample_request_low_risk.json").read_text()),
    # incomplete record: 'ca' and 'thal' unknown -> imputed by the pipeline
    {"age": 52, "sex": 1, "cp": 3, "trestbps": 172, "chol": 199, "fbs": 1, "restecg": 0,
     "thalach": 162, "exang": 0, "oldpeak": 0.5, "slope": 1},
]
pd.DataFrame(patients)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,58,1,4,140,289,0,2,120,1,1.8,2,2.0,7.0
1,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0
2,52,1,3,172,199,1,0,162,0,0.5,1,NaN,NaN


## 1. Persisted pipeline (joblib)

In [2]:
model = load_model()
meta = load_metadata()
print("Model:", meta["model_name"], "| version:", meta["model_version"], "| sklearn:", meta["sklearn_version"])
local = pd.DataFrame(predict_records(model, patients))
local

Model: logistic_regression | version: 20261011104453 | sklearn: 1.9.1


,prediction,label,probability_disease,confidence,risk_band
0,1,heart_disease,0.9964,0.9964,high
1,0,no_heart_disease,0.0358,0.9642,low
2,0,no_heart_disease,0.0978,0.9022,low


## 2. MLflow model registry
The registered model bundles the source code (`code_paths`) and pinned requirements, so it can also be served with `mlflow models serve -m models:/heart-disease-classifier@champion`.

In [3]:
mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
registered = mlflow.sklearn.load_model(f"models:/{config.REGISTERED_MODEL_NAME}@champion")
frame = pd.DataFrame(patients).reindex(columns=config.FEATURES).astype(float)
reg_proba = registered.predict_proba(frame)[:, 1]
print("Registry probabilities:", reg_proba.round(4))
assert np.allclose(reg_proba, local["probability_disease"], atol=1e-4), "registry and joblib model disagree"
print("OK - registry model and joblib artifact give identical predictions")

Registry probabilities: [0.9964 0.0358 0.0978]
OK - registry model and joblib artifact give identical predictions


## 3. REST API
Start the API first, e.g. `uvicorn heart_disease.api.main:app --port 8000` or `docker run -p 8000:8000 heart-disease-api:1.0.0`. For Kubernetes use `http://localhost:8080`.

In [4]:
API_URL = "http://localhost:8000"
try:
    print(requests.get(f"{API_URL}/health", timeout=3).json())
    for p in patients:
        r = requests.post(f"{API_URL}/predict", json=p, timeout=5)
        print(r.status_code, r.json())
    bad = requests.post(f"{API_URL}/predict", json={**patients[0], "cp": 9}, timeout=5)
    print("Invalid input ->", bad.status_code, bad.json()["detail"][0]["msg"])
except requests.ConnectionError:
    print(f"API not reachable at {API_URL} - start it to run this section.")

{'status': 'ok', 'model_loaded': True, 'model_name': 'logistic_regression', 'model_version': '20261011104453'}


200 {'prediction': 1, 'label': 'heart_disease', 'probability_disease': 0.9964, 'confidence': 0.9964, 'risk_band': 'high', 'model_name': 'logistic_regression', 'model_version': '20261011104453', 'request_id': '0073c34569434bf6b6e07370f252ae1b'}


200 {'prediction': 0, 'label': 'no_heart_disease', 'probability_disease': 0.0358, 'confidence': 0.9642, 'risk_band': 'low', 'model_name': 'logistic_regression', 'model_version': '20261011104453', 'request_id': '982a2734575c4a3da3b0b34f2eec946b'}


200 {'prediction': 0, 'label': 'no_heart_disease', 'probability_disease': 0.0978, 'confidence': 0.9022, 'risk_band': 'low', 'model_name': 'logistic_regression', 'model_version': '20261011104453', 'request_id': '3bf991e67b4b4c4b8803c553c2aa57f1'}


Invalid input -> 422 Input should be 1, 2, 3 or 4
